# v1 Mode 2 — LLM-Assisted Argument Extraction

**Purpose:** Extract structured arguments from investigation report files (DOCX/PDF, Russian or English) using an LLM, producing a JSON knowledge base matching the system's 8-field argument schema.

**Output format:** `kostenko_knowledge_base.json` structure — metadata, arguments (with `cause_categories`), and argumentation framework.

**Pipeline:**
1. Load report files (DOCX or PDF)
2. Configure source metadata per report
3. LLM extracts arguments from each report
4. Validate and assign IDs
5. LLM identifies attack/support relations across all arguments
6. Export to JSON

## 1. Setup & Imports

In [ ]:
# Run once to install dependencies
# !pip install openai python-docx PyPDF2

In [ ]:
import json
import os
import re
from pathlib import Path
from typing import Optional

from openai import OpenAI

## 2. Configuration

In [ ]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "sk-..."))
MODEL = "gpt-4o"  # or "gpt-4-turbo", "gpt-4o-mini" for cheaper runs

OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "kostenko_knowledge_base.json"

## 3. Document Loading

One function per file type. Add your report files below — the cell after defines which files to load.

In [ ]:
def load_docx(path: str) -> str:
    """Extract full text from a .docx file."""
    from docx import Document
    doc = Document(path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    # Also grab text from tables (investigation reports often use them)
    for table in doc.tables:
        for row in table.rows:
            row_text = " | ".join(cell.text.strip() for cell in row.cells if cell.text.strip())
            if row_text:
                paragraphs.append(row_text)
    return "\n".join(paragraphs)

In [ ]:
def load_pdf(path: str) -> str:
    """Extract full text from a PDF file."""
    from PyPDF2 import PdfReader
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)
    return "\n".join(pages)

In [ ]:
def load_file(path: str) -> str:
    """Auto-detect file type and load."""
    p = Path(path)
    if p.suffix.lower() == ".docx":
        return load_docx(path)
    elif p.suffix.lower() == ".pdf":
        return load_pdf(path)
    elif p.suffix.lower() == ".txt":
        return p.read_text(encoding="utf-8")
    else:
        raise ValueError(f"Unsupported file type: {p.suffix}")

### 3a. Define your report files

Edit the list below. Each entry needs:
- `file`: path to the report
- `source_id`: short ID used in argument IDs (e.g. `"U"`, `"K"`, `"D"`)
- `source_meta`: metadata about this source for the output JSON

In [ ]:
REPORTS = [
    {
        "file": "./reports/usembekov_report.docx",   # change path as necessary
        "source_id": "U",
        "source_meta": {
            "id": "U",
            "full_name": "Usembekov Meiramбек Sabdenovich",
            "affiliation": "KarTU im. Abylkas Saginov",
            "role": "Independent specialist, PhD (Technical Sciences)",
            "report_date": "2023-11"
        }
    },
    {
        "file": "./reports/kolikov_meshcheryakov_report.docx",  # change
        "source_id": "K",
        "source_meta": {
            "id": "K",
            "full_name": "Kolikov-Meshcheryakov Joint Expert Conclusion",
            "affiliation": "NUST MISIS / independent",
            "role": "Expert commission",
            "report_date": "2023-11-20"
        }
    },
    {
        "file": "./reports/dmt_report.pdf",  # change
        "source_id": "D",
        "source_meta": {
            "id": "D",
            "full_name": "DMT GmbH & Co. KG",
            "affiliation": "DMT, Essen, Germany",
            "role": "Independent technical review (contracted by ArcelorMittal)",
            "report_date": "2023-11-17"
        }
    }
]

In [ ]:
# Load all reports
report_texts = {}
for r in REPORTS:
    path = r["file"]
    sid = r["source_id"]
    try:
        text = load_file(path)
        report_texts[sid] = text
        print(f"[OK] {sid}: loaded {len(text):,} chars from {path}")
    except FileNotFoundError:
        print(f"[SKIP] {sid}: file not found — {path}")
    except Exception as e:
        print(f"[ERROR] {sid}: {e}")

print(f"\nLoaded {len(report_texts)}/{len(REPORTS)} reports.")

### 3b. (Optional) Inspect loaded text

Sanity-check that the extraction captured the content. Especially useful for PDFs with complex layouts.

In [ ]:
# Uncomment to preview first 2000 chars of a specific report
# print(report_texts["U"][:2000])

## 4. Case Metadata

Top-level metadata for the output JSON. 

Edit to match your case.

In [ ]:
CASE_METADATA = {
    "case": "Kostenko Mine Explosion",
    "date": "2023-10-28",
    "location": "Kostenko Mine, ArcelorMittal Temirtau, Kazakhstan",
    "longwall": "48K3-Z",
    "sources": [r["source_meta"] for r in REPORTS],
    "investigation_questions": [
        "Determine the cause of the fire and explosion",
        "Determine the sources of elevated methane release",
        "Determine the possible ignition source",
        "Determine the location of the fire and explosion",
        "Determine the sequence of events leading to casualties and damage",
        "Assess the role of seismic activity",
        "Propose measures to prevent similar accidents"
    ]
}

## 5. Schema & Cause Taxonomy

These are fed to the LLM so it knows the exact output format and valid `cause_categories` values.

In [ ]:
ARGUMENT_SCHEMA = {
    "id": "string — assigned later, leave as placeholder",
    "source": "string — source_id of this report",
    "topic": "string — short label: e.g. 'Ignition source', 'Methane source', 'Spontaneous combustion', 'Ventilation', 'Explosion sequence', 'Explosion type', 'Explosion location', 'Gas-dynamic event', 'Electrical equipment'",
    "claim": "string — the expert's conclusion on this topic, one clear sentence",
    "evidence": "string — specific data, measurements, observations cited to support the claim",
    "warrant": "string — the reasoning connecting the evidence to the claim",
    "confidence": "float 0.0-1.0 — how confident the expert appears (0.9+ = very certain, 0.5-0.7 = uncertain, <0.5 = speculative)",
    "cause_categories": "list of strings — one or more category IDs from the taxonomy below"
}

CAUSE_TAXONOMY = {
    "technical": [
        {"id": "TC-01", "label": "methane_accumulation", "description": "Elevated CH4 in working or goaf exceeding permissible limits"},
        {"id": "TC-02", "label": "ignition_source_mechanical", "description": "Mechanical sparking from equipment: AFC chain, shearer, angle grinder, falling metal"},
        {"id": "TC-03", "label": "ignition_source_electrical", "description": "Electrical arcing from non-explosion-proof equipment or cable damage"},
        {"id": "TC-04", "label": "ignition_source_chemical", "description": "Chemical exothermic reaction: resin, flammable aerosols, synthetic oils"},
        {"id": "TC-05", "label": "spontaneous_combustion", "description": "Endogenous fire from coal self-ignition in goaf or remnant pillars"},
        {"id": "TC-06", "label": "ventilation_failure", "description": "Failure or inadequacy of ventilation causing methane accumulation"},
        {"id": "TC-07", "label": "degasification_failure", "description": "Inadequate or absent degasification of coal seams or goaf"},
        {"id": "TC-08", "label": "geological_dynamic_hazard", "description": "Rock burst, sudden outburst, seismic event"},
        {"id": "TC-09", "label": "roof_failure", "description": "Roof fall, pillar collapse, or strata movement displacing gas"},
        {"id": "TC-10", "label": "coal_dust_hazard", "description": "Explosive coal dust accumulation providing fuel for secondary explosion"},
        {"id": "TC-11", "label": "monitoring_system_failure", "description": "Sensor failure, absence of sensors, or data not reflecting actual conditions"},
        {"id": "TC-12", "label": "equipment_technical_failure", "description": "Mechanical/structural equipment failure unrelated to ignition"},
        {"id": "TC-13", "label": "geological_structural_weakness", "description": "Geological instability: swelling clays, tectonic disturbances, weak overburden"}
    ],
    "organizational": [
        {"id": "OC-01", "label": "insufficient_production_control", "description": "Absent or inadequate internal safety supervision system"},
        {"id": "OC-02", "label": "naryad_system_violation", "description": "Work performed without written authorization or verbal modification of work orders"},
        {"id": "OC-03", "label": "qualification_training_failure", "description": "Workers admitted without required qualification or training"},
        {"id": "OC-04", "label": "data_falsification", "description": "Falsification of monitoring data or safety records"},
        {"id": "OC-05", "label": "technical_documentation_violation", "description": "Work conducted violating technical documentation"},
        {"id": "OC-06", "label": "labor_discipline_failure", "description": "Low discipline, unauthorized presence, prohibited items underground"},
        {"id": "OC-07", "label": "emergency_response_failure", "description": "Inadequate emergency preparation or response"},
        {"id": "OC-08", "label": "project_documentation_deficiency", "description": "Project docs non-compliant or failing to account for known hazards"},
        {"id": "OC-09", "label": "geotechnical_forecast_failure", "description": "Failure to conduct required geotechnical or outburst hazard forecast"},
        {"id": "OC-10", "label": "hazard_identification_failure", "description": "Failure to identify hazards associated with substances, processes, or conditions"}
    ]
}

# Flatten for the prompt
ALL_CATEGORIES = CAUSE_TAXONOMY["technical"] + CAUSE_TAXONOMY["organizational"]
TAXONOMY_TEXT = "\n".join(
    f"  {c['id']} ({c['label']}): {c['description']}" for c in ALL_CATEGORIES
)

## 6. LLM Extraction

### 6a. System prompt

This prompt instructs the LLM on what to extract and how to format it.

In [ ]:
SYSTEM_PROMPT = f"""You are an expert mining accident investigator and structured data extraction specialist.

Your task: read an investigation report and extract every distinct expert argument into a structured JSON format.

An "argument" is a distinct conclusion the expert reaches about a specific investigation topic, supported by cited evidence and reasoning. Each argument addresses ONE topic (e.g., what caused the ignition, where the explosion started, what the methane source was).

WHAT TO EXTRACT:
- Every factual conclusion the expert makes about the accident's causes, sequence, location, or mechanism
- Arguments that EXCLUDE a cause (e.g., "spontaneous combustion is excluded because...") — these are just as important
- Arguments about contributing factors (ventilation, degasification, equipment condition)
- Do NOT extract recommendations, preventive measures, or general background information

OUTPUT FORMAT — return ONLY a JSON array of objects, each with these fields:
{json.dumps(ARGUMENT_SCHEMA, indent=2)}

CAUSE CATEGORIES — assign one or more from this taxonomy:
{TAXONOMY_TEXT}

RULES:
1. Each argument must have a clear, specific claim — not vague summaries
2. Evidence must cite specific data: sensor readings, measurements, physical findings, witness statements, dates, values
3. Warrant must explain WHY the evidence supports the claim — the expert's reasoning
4. Confidence: infer from the expert's language. "Established" / "confirmed" → 0.85-0.95. "Most probable" / "likely" → 0.65-0.80. "Possible" / "cannot be excluded" → 0.40-0.60. "Unknown" → 0.30-0.50.
5. cause_categories: use the taxonomy IDs (e.g., ["TC-01", "TC-06"]). An argument can have multiple categories. An exclusion argument (e.g., "spontaneous combustion excluded") should still list the category being excluded (e.g., ["TC-05"]).
6. If the report is in Russian, extract arguments in ENGLISH. Translate technical content accurately.
7. Leave the "id" field as a placeholder string like "PLACEHOLDER" — IDs are assigned later.
8. Return ONLY the JSON array. No markdown fences, no commentary.
"""

### 6b. Extraction function

In [ ]:
def extract_arguments(report_text: str, source_id: str, model: str = MODEL) -> list[dict]:
    """
    Send report text to LLM and get back structured arguments.
    
    For very long reports (>100k chars), this splits into chunks with overlap.
    For normal-length reports, sends the full text in one call.
    """
    MAX_CHARS = 80_000  # conservative limit to leave room for prompt + response
    
    if len(report_text) <= MAX_CHARS:
        chunks = [report_text]
    else:
        # Split into overlapping chunks
        chunk_size = MAX_CHARS
        overlap = 2000
        chunks = []
        start = 0
        while start < len(report_text):
            end = start + chunk_size
            chunks.append(report_text[start:end])
            start = end - overlap
        print(f"  Report split into {len(chunks)} chunks")
    
    all_arguments = []
    
    for i, chunk in enumerate(chunks):
        user_msg = f"Source ID: {source_id}\n\n--- REPORT TEXT ---\n{chunk}\n--- END ---"
        
        if len(chunks) > 1:
            user_msg += f"\n\n(This is chunk {i+1} of {len(chunks)}. Extract all arguments found in this section.)"
        
        print(f"  Calling {model} (chunk {i+1}/{len(chunks)}, {len(chunk):,} chars)...")
        
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ],
            temperature=0.1,  # low temp for consistent extraction
            max_tokens=8000,
        )
        
        raw = response.choices[0].message.content.strip()
        
        # Clean markdown fences if the model adds them despite instructions
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        
        try:
            args = json.loads(raw)
            if isinstance(args, list):
                all_arguments.extend(args)
                print(f"  Got {len(args)} arguments from chunk {i+1}")
            else:
                print(f"  WARNING: Expected list, got {type(args)}. Skipping chunk {i+1}.")
        except json.JSONDecodeError as e:
            print(f"  ERROR parsing JSON from chunk {i+1}: {e}")
            print(f"  Raw response (first 500 chars): {raw[:500]}")
            # Save raw response for debugging
            debug_path = OUTPUT_DIR / f"debug_{source_id}_chunk{i+1}.txt"
            debug_path.write_text(raw, encoding="utf-8")
            print(f"  Saved raw response to {debug_path}")
    
    return all_arguments

### 6c. Run extraction for each report

This calls the LLM once per report. Results are stored in `extracted_args` keyed by source ID.

In [ ]:
extracted_args = {}

for sid, text in report_texts.items():
    print(f"\n{'='*60}")
    print(f"Extracting arguments from source: {sid}")
    print(f"{'='*60}")
    args = extract_arguments(text, sid)
    extracted_args[sid] = args
    print(f"\nTotal arguments from {sid}: {len(args)}")

total = sum(len(v) for v in extracted_args.values())
print(f"\n{'='*60}")
print(f"TOTAL ARGUMENTS EXTRACTED: {total}")
print(f"{'='*60}")

## 7. Validation & ID Assignment

In [ ]:
VALID_CATEGORIES = {c["id"] for c in ALL_CATEGORIES}
REQUIRED_FIELDS = {"source", "topic", "claim", "evidence", "warrant", "confidence", "cause_categories"}


def validate_argument(arg: dict, idx: int, source_id: str) -> list[str]:
    """Return list of issues found. Empty list = valid."""
    issues = []
    
    # Check required fields
    for field in REQUIRED_FIELDS:
        if field not in arg:
            issues.append(f"Missing field: {field}")
    
    # Check confidence range
    conf = arg.get("confidence")
    if conf is not None:
        if isinstance(conf, str):
            try:
                arg["confidence"] = float(conf)
            except ValueError:
                issues.append(f"confidence is not a number: {conf}")
        elif isinstance(conf, (int, float)):
            if not (0.0 <= conf <= 1.0):
                issues.append(f"confidence out of range: {conf}")
        else:
            issues.append(f"confidence has unexpected type: {type(conf)}")
    
    # Check cause_categories
    cats = arg.get("cause_categories", [])
    if isinstance(cats, list):
        invalid = [c for c in cats if c not in VALID_CATEGORIES]
        if invalid:
            issues.append(f"Invalid cause_categories: {invalid}")
    else:
        issues.append(f"cause_categories should be a list, got {type(cats)}")
    
    # Check source matches
    if arg.get("source") != source_id:
        arg["source"] = source_id  # fix silently
    
    return issues

In [ ]:
# Validate all and assign IDs
all_arguments = []
validation_issues = []

for sid, args in extracted_args.items():
    for i, arg in enumerate(args):
        issues = validate_argument(arg, i, sid)
        
        if issues:
            validation_issues.append({"source": sid, "index": i, "issues": issues, "claim": arg.get("claim", "???")})
        
        # Assign sequential ID: U-A1, U-A2, K-A1, etc.
        arg["id"] = f"{sid}-A{i+1}"
        arg["source"] = sid
        all_arguments.append(arg)

print(f"Total arguments: {len(all_arguments)}")
if validation_issues:
    print(f"\nValidation issues ({len(validation_issues)}):")
    for vi in validation_issues:
        print(f"  {vi['source']} #{vi['index']}: {vi['issues']}")
        print(f"    claim: {vi['claim'][:80]}...")
else:
    print("All arguments passed validation.")

### 7a. Preview extracted arguments

In [ ]:
# Quick summary table
for arg in all_arguments:
    cats = ", ".join(arg.get("cause_categories", []))
    print(f"{arg['id']:8s} | {arg['topic'][:30]:30s} | conf={arg.get('confidence', '?'):>5} | {cats}")

## 8. Argumentation Framework — Attack & Support Relations

This step sends ALL extracted arguments to the LLM and asks it to identify:
- **Attack relations**: where two arguments from different sources contradict each other
- **Support relations**: where arguments from different sources converge on the same conclusion
- **Open questions**: unresolved issues raised by the evidence

In [ ]:
RELATIONS_SYSTEM_PROMPT = """You are an expert in argumentation theory analyzing mining accident investigation arguments.

You will receive a list of structured arguments extracted from multiple independent expert reports about the same accident. Your task is to identify:

1. ATTACK RELATIONS — where two arguments from DIFFERENT sources reach contradictory or incompatible conclusions about the same topic.
   Types:
   - "rebutting": the claims directly contradict (mutual attack)
   - "undercutting": one claim undermines the evidence or warrant of another (directed attack)

2. SUPPORT RELATIONS — where two or more arguments from DIFFERENT sources independently reach the same or compatible conclusions.
   Strength:
   - "unanimous": all sources agree
   - "bilateral": two sources agree

3. OPEN QUESTIONS — important unresolved issues where the evidence is insufficient or conflicting.

Return ONLY a JSON object with this structure (no markdown fences, no commentary):
{
  "attack_relations": [
    {
      "id": "ATK-1",
      "attacker": "<argument id>",
      "target": "<argument id>",
      "type": "rebutting" or "undercutting",
      "description": "Explanation of why these arguments conflict"
    }
  ],
  "support_relations": [
    {
      "id": "SUP-1",
      "supporters": ["<arg id>", "<arg id>"],
      "topic": "Short label for what they agree on",
      "description": "How each argument supports the shared conclusion",
      "strength": "unanimous" or "bilateral"
    }
  ],
  "open_questions": [
    {
      "id": "OQ-1",
      "question": "The unresolved question",
      "relevance": "Why this matters for the investigation",
      "raised_by": ["<source_id>"]
    }
  ]
}

RULES:
- Only identify attacks between arguments from DIFFERENT sources
- Be specific in descriptions — cite the exact points of conflict
- An argument can participate in multiple attack and support relations
- Focus on substantive disagreements, not minor differences in wording
"""

In [ ]:
def extract_relations(arguments: list[dict], model: str = MODEL) -> dict:
    """Identify attack/support relations across all arguments."""
    # Format arguments for the prompt (compact, showing key fields)
    arg_summary = json.dumps(
        [{"id": a["id"], "source": a["source"], "topic": a["topic"], 
          "claim": a["claim"], "evidence": a["evidence"][:300], 
          "confidence": a.get("confidence")} 
         for a in arguments],
        indent=2
    )
    
    print(f"Sending {len(arguments)} arguments for relation detection...")
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": RELATIONS_SYSTEM_PROMPT},
            {"role": "user", "content": f"Here are the arguments:\n{arg_summary}"}
        ],
        temperature=0.1,
        max_tokens=4000,
    )
    
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)
    
    try:
        relations = json.loads(raw)
        n_atk = len(relations.get("attack_relations", []))
        n_sup = len(relations.get("support_relations", []))
        n_oq = len(relations.get("open_questions", []))
        print(f"Detected: {n_atk} attacks, {n_sup} supports, {n_oq} open questions")
        return relations
    except json.JSONDecodeError as e:
        print(f"ERROR parsing relations JSON: {e}")
        debug_path = OUTPUT_DIR / "debug_relations.txt"
        debug_path.write_text(raw, encoding="utf-8")
        print(f"Saved raw response to {debug_path}")
        return {"attack_relations": [], "support_relations": [], "open_questions": []}

In [ ]:
# Only run this if we have arguments from multiple sources
if len(extracted_args) >= 2:
    argumentation_framework = extract_relations(all_arguments)
else:
    print("Need arguments from at least 2 sources to detect relations.")
    print("Setting empty framework — you can fill this manually or re-run with more reports.")
    argumentation_framework = {
        "attack_relations": [],
        "support_relations": [],
        "open_questions": []
    }

### 8a. Preview relations

In [ ]:
print("ATTACK RELATIONS:")
for atk in argumentation_framework.get("attack_relations", []):
    print(f"  {atk['id']}: {atk['attacker']} --({atk['type']})--> {atk['target']}")
    print(f"    {atk['description'][:100]}...")

print("\nSUPPORT RELATIONS:")
for sup in argumentation_framework.get("support_relations", []):
    print(f"  {sup['id']}: {sup['supporters']} — {sup['topic']} [{sup.get('strength', '?')}]")

print("\nOPEN QUESTIONS:")
for oq in argumentation_framework.get("open_questions", []):
    print(f"  {oq['id']}: {oq['question'][:80]}...")

## 9. Compile & Export

Assemble the full knowledge base JSON and save.

In [ ]:
knowledge_base = {
    "metadata": CASE_METADATA,
    "arguments": all_arguments,
    "argumentation_framework": argumentation_framework
}

# Save
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(knowledge_base, f, indent=2, ensure_ascii=False)

print(f"Knowledge base saved to: {OUTPUT_FILE}")
print(f"  Sources: {len(CASE_METADATA['sources'])}")
print(f"  Arguments: {len(all_arguments)}")
print(f"  Attack relations: {len(argumentation_framework.get('attack_relations', []))}")
print(f"  Support relations: {len(argumentation_framework.get('support_relations', []))}")
print(f"  Open questions: {len(argumentation_framework.get('open_questions', []))}")

## 10. (Optional) Compare Against Ground Truth

If you have a manually-created KB (like the existing `kostenko_knowledge_base.json`), you can compare the LLM extraction against it.

In [ ]:
GROUND_TRUTH_PATH = None  # Set to path of your manual KB, e.g. "./kostenko_knowledge_base.json"

if GROUND_TRUTH_PATH:
    with open(GROUND_TRUTH_PATH, "r", encoding="utf-8") as f:
        gt = json.load(f)
    
    gt_args = gt["arguments"]
    ext_args = all_arguments
    
    print(f"Ground truth: {len(gt_args)} arguments")
    print(f"Extracted:    {len(ext_args)} arguments")
    
    # Compare by topic coverage
    gt_topics = {a["topic"] for a in gt_args}
    ext_topics = {a["topic"] for a in ext_args}
    
    print(f"\nTopics in ground truth: {sorted(gt_topics)}")
    print(f"Topics in extracted:    {sorted(ext_topics)}")
    print(f"\nMissing topics: {gt_topics - ext_topics}")
    print(f"Extra topics:   {ext_topics - gt_topics}")
    
    # Compare by source
    for sid in set(a["source"] for a in gt_args):
        gt_count = len([a for a in gt_args if a["source"] == sid])
        ext_count = len([a for a in ext_args if a["source"] == sid])
        print(f"\nSource {sid}: ground truth={gt_count}, extracted={ext_count}")
else:
    print("No ground truth path set. Skipping comparison.")